In [ ]:
# Run from the repo root so relative imports and `!`-shell commands resolve.
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


# GradCAM: random / clean-trained / poison-trained

Each row = one sample. Columns are configured via `COLUMNS` in the Config cell — comment out any line to drop that column. Supported entries:

- `'input'`             — raw test image $x$
- `'reference'`         — same-identity anchor image $a$
- `('random',  'clean')`     — random-init model on $x$
- `('clean',   'clean')`     — clean model on $x$
- `('clean',   'poisoned')`  — clean model on $x{+}\delta$
- `('poison',  'clean')`     — poison model on $x$
- `('poison',  'poisoned')`  — poison model on $x{+}\delta$
- `('poison',  'delta')`     — poison model on $\delta$

GradCAM target: cosine to per-model same-identity anchor embedding. Required models / anchor sampling are inferred automatically from `COLUMNS`.

In [ ]:
import os, io, pickle
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import transforms
import mxnet as mx
import matplotlib.pyplot as plt

from backbones import get_model
from backbones.custom_generator import ResNetGenerator

## Config

In [ ]:
# TODO: local paths in this cell were redacted to ""; set them before running.
GEN_CKPT          = ""
CKPT_CLEAN        = ""     # R-50 trained on D_c
CKPT_POISON       = ""    # R-50 trained on D_u

TEST_SOURCE       = 'lfw'
DATA_ROOT_CASIA   = ""
LFW_BIN           = ""

ARCH              = 'r50'

# 2-row layout: first ceil(N/2) rows are same-identity (test, anchor) pairs,
# remaining rows are different-identity pairs. With N_SAMPLES=2: row 0 same, row 1 diff.
N_SAMPLES         = 2
ROW_LABELS        = ['Same ID', 'Diff. ID']

GRADCAM_TARGET    = 'cosine'   # 'cosine' (recommended) | 'norm'
USE_GRADCAM_PP    = True
TARGET_LAYER_NAME = 'layer4'   # try 'layer3' for finer spatial detail

DELTA_VIS_SCALE   = 1.0        # 1.0 = same scale as other images (almost gray)
OVERLAY_ALPHA     = 0.45

# ---------------------------------------------------------------- columns
# Comment out any line to drop that column.
COLUMNS = [
    'input',
    'reference',
#     ('random',  'clean'),
    ('clean',   'clean'),
#     ('clean',   'poisoned'),
    ('poison',  'clean'),
    ('poison',  'poisoned'),
    ('poison',  'delta'),
]

OUT_PATH          = "gradcam_figure.pdf"
SEED              = 707719
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(SEED); np.random.seed(SEED)

# ---- derived ----
needed_model_keys = {c[0] for c in COLUMNS if isinstance(c, tuple)}
need_anchor = ('reference' in COLUMNS) or (GRADCAM_TARGET == 'cosine')
print('device:', DEVICE,
      ' models needed:', needed_model_keys,
      ' anchor needed:', need_anchor,
      ' columns:', len(COLUMNS),
      ' rows:', N_SAMPLES, '(same+diff split)')

## Test sample loaders

In [ ]:
from eval.verification import load_bin


# ============================================================
# LFW pair cache — load_bin runs ONCE per session.
# ============================================================
if 'lfw_pairs_cache' not in globals():
    print('Loading LFW (one-time)...')
    _data, _issame = load_bin(LFW_BIN, (112, 112))
    _imgs_norm = (_data[0] - 127.5) / 127.5     # [N, 3, 112, 112] in [-1, 1]
    lfw_pairs_cache = [
        (_imgs_norm[2 * i], _imgs_norm[2 * i + 1], int(_issame[i]))
        for i in range(len(_issame))
    ]
    print(f'  cached {len(lfw_pairs_cache)} LFW pairs')
else:
    print(f'LFW pairs already cached: {len(lfw_pairs_cache)} pairs')


def load_lfw_pairs(path=None, image_size=None):
    """Return cached LFW pair list of (test, anchor, is_same)."""
    return lfw_pairs_cache


class MXFaceDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir):
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
        ])
        self.imgrec = mx.recordio.MXIndexedRecordIO(
            os.path.join(root_dir, 'train.idx'),
            os.path.join(root_dir, 'train.rec'), 'r',
        )
        s = self.imgrec.read_idx(0)
        header, _ = mx.recordio.unpack(s)
        if header.flag > 0:
            self.imgidx = np.array(range(1, int(header.label[0])))
        else:
            self.imgidx = np.array(list(self.imgrec.keys))

    def __len__(self):
        return len(self.imgidx)

    def get_by_recidx(self, idx):
        s = self.imgrec.read_idx(int(idx))
        _, img_bytes = mx.recordio.unpack(s)
        img = mx.image.imdecode(img_bytes).asnumpy()
        return self.transform(img)

    def __getitem__(self, index):
        return self.get_by_recidx(self.imgidx[index])


def build_id_index(ds):
    id2idx = {}
    for idx in ds.imgidx:
        s = ds.imgrec.read_idx(int(idx))
        header, _ = mx.recordio.unpack(s)
        label = header.label
        if isinstance(label, (np.ndarray, list)):
            label = int(label[0])
        else:
            label = int(label)
        id2idx.setdefault(label, []).append(int(idx))
    return id2idx


def sample_test_inputs(n, need_anchor, seed=None):
    """Return (test_imgs, anchor_imgs) of shape [n, 3, 112, 112].

    Layout: first ceil(n/2) rows are same-identity pairs (test, anchor share id),
    remaining floor(n/2) rows are different-identity pairs.

    Uses a *local* RNG keyed by `seed` — no global RNG side-effects, so
    iterating over seeds doesn't perturb other code.
    """
    rng = np.random.default_rng(seed)
    n_same = (n + 1) // 2
    n_diff = n - n_same

    if TEST_SOURCE == 'lfw':
        pairs = load_lfw_pairs()              # cached
        same = [p for p in pairs if p[2] == 1]
        diff = [p for p in pairs if p[2] == 0]
        rng.shuffle(same); rng.shuffle(diff)
        sel = same[:n_same] + diff[:n_diff]
        test = torch.stack([p[0] for p in sel]).to(DEVICE)
        anch = (torch.stack([p[1] for p in sel]).to(DEVICE)
                if need_anchor else None)
        return test, anch

    if TEST_SOURCE == 'casia':
        ds = MXFaceDataset(DATA_ROOT_CASIA)
        if not need_anchor:
            idx = rng.permutation(len(ds))[:n]
            test = torch.stack([ds[int(i)] for i in idx]).to(DEVICE)
            return test, None

        print('  building identity index ...', flush=True)
        id2idx = build_id_index(ds)
        valid = [i for i, lst in id2idx.items() if len(lst) >= 2]

        # Same-id rows: 2 imgs from one identity → (test, anchor)
        chosen_same = rng.choice(valid, size=n_same, replace=False)
        test_l, anch_l = [], []
        for cid in chosen_same:
            p = rng.choice(id2idx[int(cid)], size=2, replace=False)
            test_l.append(ds.get_by_recidx(int(p[0])))
            anch_l.append(ds.get_by_recidx(int(p[1])))

        # Diff-id rows: 1 img from id_a (test), 1 img from id_b (anchor), a≠b
        all_ids = list(id2idx.keys())
        for _ in range(n_diff):
            a, b = rng.choice(all_ids, size=2, replace=False)
            test_l.append(ds.get_by_recidx(int(rng.choice(id2idx[int(a)]))))
            anch_l.append(ds.get_by_recidx(int(rng.choice(id2idx[int(b)]))))

        return (torch.stack(test_l).to(DEVICE),
                torch.stack(anch_l).to(DEVICE))

    raise ValueError(f'unknown TEST_SOURCE: {TEST_SOURCE}')

## Load generator

In [ ]:
if 'G' not in globals():
    G = ResNetGenerator().to(DEVICE).eval()
    state = torch.load(GEN_CKPT, map_location=DEVICE)
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    state = {k.replace('module.', ''): v for k, v in state.items()}
    missing, unexpected = G.load_state_dict(state, strict=False)
    print('Generator loaded.  missing/unexpected:', len(missing), len(unexpected))
else:
    print('Generator already loaded.')

## Load only the backbones referenced by `COLUMNS`

In [ ]:
def load_backbone(ckpt_path):
    model = get_model(ARCH, fp16=False)
    state = torch.load(ckpt_path, map_location='cpu')
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    state = {k.replace('module.', ''): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f'  missing={len(missing)}  unexpected={len(unexpected)}')
    return model.to(DEVICE).eval()


# Idempotent backbone cache — only loads what's needed and not yet loaded.
if 'models' not in globals():
    models = {}

if 'clean' in needed_model_keys and 'clean' not in models:
    print('loading clean R-50 ...')
    models['clean'] = load_backbone(CKPT_CLEAN)
if 'poison' in needed_model_keys and 'poison' not in models:
    print('loading poison R-50 ...')
    models['poison'] = load_backbone(CKPT_POISON)
if 'random' in needed_model_keys and 'random' not in models:
    print('loading random-init R-50 ...')
    models['random'] = get_model(ARCH, fp16=False).to(DEVICE).eval()

print('models:', list(models.keys()))

## Sample inputs (+ anchors), build clean / poisoned / δ

In [ ]:
test_imgs, anch_imgs = sample_test_inputs(N_SAMPLES, need_anchor, seed=SEED)
print('test:', test_imgs.shape,
      ' anchor:', None if anch_imgs is None else anch_imgs.shape)

anchors = None
if GRADCAM_TARGET == 'cosine' and anch_imgs is not None:
    anchors = {}
    with torch.no_grad():
        for k, m in models.items():
            anchors[k] = F.normalize(m(anch_imgs).float(), dim=1)
    print('anchor embeddings:', {k: v.shape for k, v in anchors.items()})

with torch.no_grad():
    delta = G(test_imgs)
    poison_imgs = (test_imgs + delta).clamp(-1, 1)

inputs = {'clean': test_imgs, 'poisoned': poison_imgs, 'delta': delta}

## GradCAM (with optional GradCAM++)

In [ ]:
def get_target_layer(model, name=None):
    if name is None:
        last = None
        for n, m in model.named_modules():
            if isinstance(m, torch.nn.Conv2d):
                last = (n, m)
        if last is None:
            raise ValueError('no Conv2d found')
        return last
    for n, m in model.named_modules():
        if n == name:
            return (n, m)
    raise ValueError(f'layer "{name}" not in model')


class GradCAMUnified:
    def __init__(self, model, target_layer, mode='cosine', anchor=None, use_pp=True):
        self.model = model
        self.mode = mode
        self.anchor = anchor.detach() if anchor is not None else None
        self.use_pp = use_pp
        self.activations = None
        self.gradients = None
        self._h1 = target_layer.register_forward_hook(self._sa)
        self._h2 = target_layer.register_full_backward_hook(self._sg)

    def _sa(self, m, i, o):
        self.activations = o.detach()

    def _sg(self, m, gi, go):
        self.gradients = go[0].detach()

    def remove(self):
        self._h1.remove(); self._h2.remove()

    def __call__(self, x):
        x = x.clone().detach().to(DEVICE).requires_grad_(True)
        self.model.zero_grad()
        emb = self.model(x).float()
        if self.mode == 'cosine':
            emb_n = F.normalize(emb, dim=1)
            target = (emb_n * self.anchor).sum()
        elif self.mode == 'norm':
            target = emb.norm(dim=1).sum()
        else:
            raise ValueError(self.mode)
        target.backward()

        grad, act = self.gradients, self.activations
        if self.use_pp:
            grad2 = grad.pow(2)
            grad3 = grad.pow(3)
            sum_ag3 = (act * grad3).sum(dim=(2, 3), keepdim=True)
            alpha = grad2 / (2 * grad2 + sum_ag3 + 1e-8)
            weights = (alpha * F.relu(grad)).sum(dim=(2, 3), keepdim=True)
        else:
            weights = grad.mean(dim=(2, 3), keepdim=True)

        cam = F.relu((weights * act).sum(dim=1))
        N = cam.size(0)
        flat = cam.view(N, -1)
        cmin = flat.min(dim=1)[0].view(N, 1, 1)
        cmax = flat.max(dim=1)[0].view(N, 1, 1)
        cam = (cam - cmin) / (cmax - cmin + 1e-8)
        cam = F.interpolate(cam.unsqueeze(1), size=x.shape[-2:],
                            mode='bilinear', align_corners=False).squeeze(1)
        return cam.detach().cpu().numpy()

## Run GradCAM only for tuple columns

In [ ]:
gc_specs = [c for c in COLUMNS if isinstance(c, tuple)]

cams = {}
for mk, ik in gc_specs:
    name, layer = get_target_layer(models[mk], TARGET_LAYER_NAME)
    print(f'({mk:>6}, {ik:>8}): target = {name}')
    anch = anchors[mk] if anchors is not None else None
    g = GradCAMUnified(models[mk], layer,
                       mode=GRADCAM_TARGET, anchor=anch,
                       use_pp=USE_GRADCAM_PP)
    cams[(mk, ik)] = g(inputs[ik])
    g.remove()

## Visualize

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

def img_to_disp(t):
    x = t.detach().cpu().numpy()
    return np.clip(((x + 1.0) / 2.0).transpose(1, 2, 0), 0, 1)

def delta_to_disp(t, scale=DELTA_VIS_SCALE):
    x = t.detach().cpu().numpy() * scale
    return np.clip(((x + 1.0) / 2.0).transpose(1, 2, 0), 0, 1)

def overlay(img_disp, cam, alpha=OVERLAY_ALPHA):
    cmap = plt.get_cmap('jet')
    cam_rgb = cmap(cam)[..., :3]
    return np.clip((1 - alpha) * img_disp + alpha * cam_rgb, 0, 1)

TITLES = {
    'input':                 'Input\n' + r'$x$',
    'reference':             'Reference\n' + r'$x^{\prime}$',
    ('random',  'clean'):    'Random init\n' + r'Clean $x$',
    ('clean',   'clean'):    'Clean Model\n' + r'Clean $x$',
    ('clean',   'poisoned'): 'Clean Model\n' + r'Poisoned $x{+}\delta$',
    ('poison',  'clean'):    'Poison Model\n' + r'Clean $x$',
    ('poison',  'poisoned'): 'Poison Model\n' + r'Poisoned $x{+}\delta$',
    ('poison',  'delta'):    'Poison Model\n' + r'Noise $\delta$',
}

def render_cell(ax, spec, row):
    if spec == 'input':
        ax.imshow(img_to_disp(test_imgs[row]))
    elif spec == 'reference':
        if anch_imgs is not None:
            ax.imshow(img_to_disp(anch_imgs[row]))
        else:
            ax.set_facecolor('lightgray')
    else:
        mk, ik = spec
        x = inputs[ik][row]
        disp = delta_to_disp(x) if ik == 'delta' else img_to_disp(x)
        ax.imshow(overlay(disp, cams[spec][row]))
    ax.set_xticks([]); ax.set_yticks([])

N = N_SAMPLES
n_cols = len(COLUMNS)
cell = 1.0
fig, axes = plt.subplots(N, n_cols,
                         figsize=(cell * n_cols, cell * N + 0.5),
                         dpi=200, squeeze=False)

for r in range(N):
    for c, spec in enumerate(COLUMNS):
        render_cell(axes[r, c], spec, r)
        if r == 0:
            axes[r, c].set_title(TITLES.get(spec, str(spec)),
                                 fontsize=10, pad=2)
    # Row label on the leftmost column (Same ID / Diff. ID)
    if r < len(ROW_LABELS):
        axes[r, 0].set_ylabel(ROW_LABELS[r], fontsize=10, labelpad=4)

for ax in axes.ravel():
    for s in ax.spines.values():
        s.set_linewidth(0.4)

plt.tight_layout(pad=0.3, h_pad=0.25, w_pad=0.25)
fig.savefig(OUT_PATH, bbox_inches='tight')
fig.savefig(OUT_PATH.replace('.pdf', '.png'), bbox_inches='tight', dpi=500)
plt.show()
print('Saved', OUT_PATH)

## Seed exploration

Cached G + backbones + LFW. Each iteration re-samples, recomputes anchors / δ / GradCAMs, and re-renders inline. Edit `SEED_LIST` and re-run; copy the best seed back into `SEED` (config) for the canonical save.

In [ ]:
SEED_LIST = [707719, 8, 42, 100, 1234, 2024]
SEED_LIST = [8]
SAVE_PER_SEED = True   # True → also writes ./gradcam_figure_seed{S}.pdf for each


def render_gradcam_for_seed(seed, save=False, show=True, dpi=400):
    """End-to-end: re-sample → anchors → δ → GradCAM → render. Uses cached G/models."""
    test_s, anch_s = sample_test_inputs(N_SAMPLES, need_anchor, seed=seed)

    anchors_s = None
    if GRADCAM_TARGET == 'cosine' and anch_s is not None:
        anchors_s = {}
        with torch.no_grad():
            for k, m in models.items():
                anchors_s[k] = F.normalize(m(anch_s).float(), dim=1)

    with torch.no_grad():
        delta_s = G(test_s)
        poison_s = (test_s + delta_s).clamp(-1, 1)
    inputs_s = {'clean': test_s, 'poisoned': poison_s, 'delta': delta_s}

    # GradCAM only for tuple columns
    cams_s = {}
    for mk, ik in [c for c in COLUMNS if isinstance(c, tuple)]:
        _, layer = get_target_layer(models[mk], TARGET_LAYER_NAME)
        anch_e = anchors_s[mk] if anchors_s is not None else None
        g = GradCAMUnified(models[mk], layer,
                           mode=GRADCAM_TARGET, anchor=anch_e,
                           use_pp=USE_GRADCAM_PP)
        cams_s[(mk, ik)] = g(inputs_s[ik])
        g.remove()

    def cell(ax, spec, row):
        if spec == 'input':
            ax.imshow(img_to_disp(test_s[row]))
        elif spec == 'reference':
            if anch_s is not None:
                ax.imshow(img_to_disp(anch_s[row]))
            else:
                ax.set_facecolor('lightgray')
        else:
            mk, ik = spec
            t = inputs_s[ik][row]
            disp = delta_to_disp(t) if ik == 'delta' else img_to_disp(t)
            ax.imshow(overlay(disp, cams_s[spec][row]))
        ax.set_xticks([]); ax.set_yticks([])

    n_cols = len(COLUMNS)
    fig, axes = plt.subplots(N_SAMPLES, n_cols,
                             figsize=(n_cols + 0.5, N_SAMPLES + 0.5),
                             dpi=dpi, squeeze=False)
    for r in range(N_SAMPLES):
        for c, spec in enumerate(COLUMNS):
            cell(axes[r, c], spec, r)
            if r == 0:
                axes[r, c].set_title(TITLES.get(spec, str(spec)),
                                     fontsize=10, pad=2)
        if r < len(ROW_LABELS):
            axes[r, 0].set_ylabel(ROW_LABELS[r], fontsize=10, labelpad=4)
    for ax in axes.ravel():
        for sp in ax.spines.values():
            sp.set_linewidth(0.4)

#     fig.suptitle(f'seed = {seed}', fontsize=8, y=0.995)
    plt.tight_layout(pad=0.5, h_pad=0.25, w_pad=0.5)

    if save:
        out = OUT_PATH.replace('.pdf', f'_seed{seed}.pdf')
        fig.savefig(out, bbox_inches='tight')
        fig.savefig(out.replace('.pdf', '.png'), bbox_inches='tight', dpi=300)
        print(f'  saved {out}')
    if show:
        plt.show()
    else:
        plt.close(fig)


for s in SEED_LIST:
    render_gradcam_for_seed(s, save=SAVE_PER_SEED, show=True)